In [ ]:
# === ALL-IN-ONE VTU EXAMMATE AI (PASTE INTO ONE COLAB CELL & RUN) ===

# 1. Install Python packages, zstd, and Ollama
!pip install -q chromadb gradio ollama pypdf
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os
import time
import subprocess
import chromadb
import ollama
import gradio as gr
from pypdf import PdfReader

# 2. Start Ollama server in the background and wait until ready
subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(20):
    try:
        ollama.list()
        break
    except Exception:
        time.sleep(1)

# 3. Download local AI models (no API key needed)
!/usr/local/bin/ollama pull llama3.2:1b
!/usr/local/bin/ollama pull nomic-embed-text

# 4. Vector database & PDF RAG pipeline
chroma_client = chromadb.PersistentClient(path="./vtu_exammate_chroma")


def get_or_create_collection():
    try:
        return chroma_client.get_collection("vtu_notes")
    except Exception:
        return chroma_client.create_collection("vtu_notes")


def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text_data = [page.extract_text() for page in reader.pages if page.extract_text()]
    return "\n".join(text_data)


def split_text_into_chunks(text, chunk_size=800, overlap=150):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[i : i + chunk_size]))
        if i + chunk_size >= len(words):
            break
    return chunks


def process_document_to_db(pdf_file_path):
    if not pdf_file_path:
        return "❌ Error: Please upload a PDF file first!"
    try:
        raw_text = extract_text_from_pdf(pdf_file_path)
        if not raw_text.strip():
            return "❌ Error: The uploaded PDF is empty or contains no extractable text."

        chunks = split_text_into_chunks(raw_text)
        if not chunks:
            return "❌ Error: Could not chunk the extracted text."

        try:
            chroma_client.delete_collection("vtu_notes")
        except Exception:
            pass

        collection = chroma_client.create_collection("vtu_notes")
        embeddings = ollama.embed(model="nomic-embed-text", input=chunks)["embeddings"]
        ids = [f"chunk_{i}" for i in range(len(chunks))]
        collection.add(ids=ids, documents=chunks, embeddings=embeddings)

        return f"✅ Success! Indexed {len(chunks)} chunks from '{os.path.basename(pdf_file_path)}'."
    except Exception as e:
        return f"❌ Processing Error: {str(e)}"


def query_rag(question, task_type="general"):
    collection = get_or_create_collection()
    if collection.count() == 0:
        return "📚 Please upload and process a PDF first before asking questions."
    if not question or not question.strip():
        return "⚠️ Please enter a valid question or topic."

    try:
        question_embed = ollama.embed(model="nomic-embed-text", input=question)["embeddings"][0]
        results = collection.query(
            query_embeddings=[question_embed],
            n_results=min(4, collection.count()),
        )
        context = "\n\n---\n\n".join(results["documents"][0])

        if task_type == "summary":
            system_prompt = "You are an expert VTU Engineering Professor. Summarize the provided context clearly in bullet points."
        elif task_type == "questions_short":
            system_prompt = "Generate 5 high-yield short-answer conceptual questions based ONLY on the context below. Provide direct answers for each."
        elif task_type == "questions_long":
            system_prompt = "Generate 3 exam-style long-answer questions (8-10 marks format as per VTU norms) based ONLY on the context below, with structured evaluation key points."
        else:
            system_prompt = (
                "You are an expert VTU Engineering Subject Matter Assistant. "
                "Answer the student's question clearly based ONLY on the context provided below. "
                "If the answer is not in the context, respond with: "
                "'This information is not explicitly found in the uploaded academic notes.'"
            )

        response = ollama.chat(
            model="llama3.2:1b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion/Instruction:\n{question}"},
            ],
        )
        return response["message"]["content"]
    except Exception as e:
        return f"❌ RAG Error: {str(e)}"


def gradio_chat_handler(message, history, task):
    task_mapping = {
        "Standard RAG Chat": "general",
        "Summarize Context": "summary",
        "Generate Short Questions": "questions_short",
        "Generate Long Exam Questions": "questions_long",
    }
    return query_rag(message, task_type=task_mapping.get(task, "general"))


# 5. Build and launch Gradio app
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown(
        """
        # 📚 VTU ExamMate AI
        ### **Intelligent Academic Study Assistant**
        *Upload your VTU PDF notes or syllabus, click Process Document, and start studying.*
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📁 Upload Study Notes")
            pdf_input = gr.File(label="Upload PDF Notes / Syllabus", file_types=[".pdf"])
            process_btn = gr.Button("⚙️ Process Document", variant="primary")
            status_output = gr.Textbox(
                label="System Status",
                interactive=False,
                placeholder="Upload file & click Process...",
            )

        with gr.Column(scale=2):
            gr.Markdown("### 💬 Ask and Study")
            study_mode = gr.Radio(
                choices=[
                    "Standard RAG Chat",
                    "Summarize Context",
                    "Generate Short Questions",
                    "Generate Long Exam Questions",
                ],
                value="Standard RAG Chat",
                label="Study Assistant Mode",
            )
            gr.ChatInterface(
                fn=gradio_chat_handler,
                additional_inputs=[study_mode],
                examples=[
                    ["Explain the core concept of this chapter.", "Standard RAG Chat"],
                    ["Give me a summary of this document.", "Summarize Context"],
                    ["Generate important short questions.", "Generate Short Questions"],
                    ["Generate 10-mark VTU exam questions.", "Generate Long Exam Questions"],
                ],
            )

    process_btn.click(
        fn=process_document_to_db,
        inputs=[pdf_input],
        outputs=[status_output],
    )

demo.launch(share=True, debug=True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-d